# 04 - Stopping-rule zoo (decision gate G2)
Every post-hoc rule is evaluated at every threshold in one pass over each test set (thresholds
chosen on ID-val, `src/glwm2/readouts/rules.py`); the jointly trained gates (ACT, PonderNet,
joint correctness head, v1 GLWM gate) come from their own runs. Checks **H1** (length leakage of
pooled gates) and **H2** (premature stopping of confidence/convergence rules), then **G2**.

In [1]:
# parameters
CONFIGS = ["configs/main_addition.yaml", "configs/main_parity.yaml", "configs/main_gates.yaml"]
TASKS = ["addition", "parity"]
SPLITS = ["id_test", "ood_depth", "ood_length", "ood_both"]

In [2]:
%matplotlib inline
import json, sys, time
from pathlib import Path
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from glwm2 import PROJECT_ROOT, RESULTS_DIR, PAPER_DIR
from glwm2.eval.plotting import set_style
set_style()
WRITTEN = []  # files this notebook writes

In [3]:
# Gate tuning (equal budget, addition seed 0): pick per method the setting with the fewest ID-val
# steps among those within 0.5 points of the best ID-val accuracy. The choice is copied into
# configs/main_gates.yaml before the main gate runs.
from glwm2.eval.results import configs_from_yaml, finished, metric_rows
from glwm2.utils.io import write_json
GATE_RULE = {"act": "act", "ponder": "ponder", "v1gate": "gate"}
HP = {"act": ["act_tau"], "ponder": ["ponder_beta", "ponder_lambda_p"], "v1gate": ["v1_lambda"]}
tune = [c for f in ["configs/tune_gates.yaml", "configs/main_gates.yaml"] for c in configs_from_yaml(PROJECT_ROOT / f)
        if c.task == "addition" and c.seed == 0]
tdf = metric_rows(finished(tune))
tuning = {}
if len(tdf):
    tdf = tdf[np.array([GATE_RULE.get(m) == r for m, r in zip(tdf.method, tdf.rule)]) & (tdf.split == "id_val").to_numpy()]
    for m, sub in tdf.groupby("method"):
        ok = sub[sub.acc >= sub.acc.max() - 0.005].sort_values("mean_steps")
        best = ok.iloc[0]
        tuning[m] = {k: float(best[k]) for k in HP[m]}
        display(sub[HP[m] + ["acc", "mean_steps"]].round(4))
print("selected gate settings:", tuning)
WRITTEN.append(write_json(RESULTS_DIR / "gates" / "tuning.json", tuning))

,act_tau,acc,mean_steps
11,0.01,0.9980,2.310
107,0.10,0.9355,1.819
203,0.01,0.9980,2.310


,ponder_beta,ponder_lambda_p,acc,mean_steps
300,0.01,0.1,1.0,4.8785


,v1_lambda,acc,mean_steps
397,0.01,0.9975,2.3095


selected gate settings: {'act': {'act_tau': 0.01}, 'ponder': {'ponder_beta': 0.01, 'ponder_lambda_p': 0.1}, 'v1gate': {'v1_lambda': 0.01}}


In [4]:
from glwm2.eval.results import configs_from_yaml, finished, curve_rows, step_acc
from glwm2.eval.analysis import (ENTRIES, BASELINES, OURS, POOLED_JOINT, CONF_RULES, entry_rows, entry_instances,
                                 summary_table, g2_check, g3_check, binned, premature_ci, length_leakage_ci)
from glwm2.eval.plotting import savefig, FAMILY_COLORS
from glwm2.utils.io import write_json
cfgs = {t: [c for f in CONFIGS for c in configs_from_yaml(PROJECT_ROOT / f) if c.task == t] for t in TASKS}
for t in TASKS:
    print(t, f"{len(finished(cfgs[t]))}/{len(cfgs[t])} runs finished, seeds:",
          sorted({c.seed for c in finished(cfgs[t])}))
rows = {t: entry_rows(cfgs[t]) for t in TASKS}

addition 14/14 runs finished, seeds: [0, 1]
parity 14/14 runs finished, seeds: [0, 1]


In [5]:
for t in TASKS:
    print(f"== {t}: accuracy (mean over seeds)")
    display(summary_table(rows[t], SPLITS, "acc")["mean"].round(3))

== addition: accuracy (mean over seeds)


split,id_test,ood_both,ood_depth,ood_length
entry,,,,
fixed_T,1.000,0.495,0.709,1.000
oracle_lb,1.000,0.996,1.000,1.000
oracle_stable,1.000,0.996,1.000,1.000
oracle_iter,1.000,1.000,1.000,1.000
max_softmax,1.000,0.998,1.000,1.000
entropy,1.000,0.996,1.000,1.000
stability,0.999,0.998,0.999,1.000
kl,1.000,0.998,0.998,1.000
hidden_change,1.000,0.427,1.000,0.695


== parity: accuracy (mean over seeds)


split,id_test,ood_both,ood_depth,ood_length
entry,,,,
fixed_T,1.000,0.000,0.309,1.000
oracle_lb,1.000,0.202,0.797,1.000
oracle_stable,1.000,0.854,1.000,1.000
oracle_iter,1.000,0.854,1.000,1.000
max_softmax,1.000,0.008,0.510,1.000
entropy,1.000,0.000,0.352,1.000
stability,1.000,0.045,0.510,1.000
kl,1.000,0.566,0.940,1.000
hidden_change,1.000,0.000,0.632,0.986


In [6]:
for t in TASKS:
    print(f"== {t}: premature-stop rate and mean steps on OOD-depth / OOD-both")
    sub = rows[t][rows[t].split.isin(["ood_depth", "ood_both"])]
    display(sub.pivot_table(index="entry", columns="split", values=["premature", "mean_steps", "mean_lb"], aggfunc="mean")
            .reindex([e for e in ENTRIES if e in set(sub.entry)]).round(3))

== addition: premature-stop rate and mean steps on OOD-depth / OOD-both


mean_lb           mean_steps           premature          
split           ood_both ood_depth   ood_both ood_depth  ood_both ood_depth
entry                                                                      
fixed_T           75.356    13.496     10.000    10.000     1.000     1.000
oracle_lb         75.356    13.496     75.356    13.496     0.000     0.000
oracle_stable     75.356    13.496     40.332     4.881     0.491     0.730
oracle_iter       75.356    13.496     39.510     4.226     0.497     0.770
max_softmax       75.356    13.496     75.561    13.496     0.000     0.000
entropy           75.356    13.496     75.365    13.496     0.000     0.000
stability         75.356    13.496     40.474     5.227     0.497     0.746
kl                75.356    13.496     40.619     5.320     0.497     0.746
hidden_change     75.356    13.496      4.185    14.081     1.000     0.000
max_conf_fan      75.356    13.496    144.000    36.000     0.000     0.000
trm_post          75.356    13.496     50.035     5.382     0.996     0.753
act               75.356    13.496      2.994     2.440     1.000     1.000
ponder            75.356    13.496      5.546     6.471     1.000     0.751
trm_joint         75.356    13.496     15.106     4.248     1.000     0.746
v1gate            75.356    13.496      4.008     2.102     1.000     1.000
done_post         75.356    13.496     75.356    13.496     0.000     0.000
done_prop_post    75.356    13.496     75.356    13.490     0.000     0.006
done_joint        75.356    13.496     75.356    13.496     0.000     0.000
done_prop_joint   75.356    13.496     75.356    13.496     0.000     0.000

== parity: premature-stop rate and mean steps on OOD-depth / OOD-both


mean_lb           mean_steps           premature          
split           ood_both ood_depth   ood_both ood_depth  ood_both ood_depth
entry                                                                      
fixed_T           75.255    13.335      9.000     9.000     1.000     1.000
oracle_lb         75.255    13.335     75.255    13.335     0.000     0.000
oracle_stable     75.255    13.335     90.068    11.800     0.187     0.592
oracle_iter       75.255    13.335     90.060    11.791     0.187     0.592
max_softmax       75.255    13.335     37.445    10.286     0.982     0.908
entropy           75.255    13.335     17.884     9.411     1.000     0.992
stability         75.255    13.335     54.106    10.747     0.855     0.862
kl                75.255    13.335     83.898    12.984     0.285     0.492
hidden_change     75.255    13.335     13.242    11.039     1.000     0.811
max_conf_fan      75.255    13.335    144.000    36.000     0.000     0.000
trm_post          75.255    13.335      8.546     9.071     1.000     1.000
act               75.255    13.335     52.353    11.376     1.000     0.976
ponder            75.255    13.335     58.890    13.110     0.922     0.303
trm_joint         75.255    13.335     26.972    11.277     1.000     0.926
v1gate            75.255    13.335    144.000    36.000     0.000     0.000
done_post         75.255    13.335      8.971     8.970     1.000     1.000
done_prop_post    75.255    13.335      8.994     8.978     1.000     1.000
done_joint        75.255    13.335      8.890     8.992     1.000     1.000
done_prop_joint   75.255    13.335      8.970     8.996     1.000     1.000

In [7]:
# H1: length leakage (partial Spearman of steps with length given depth) on ID-test + OOD-length,
# and whether pooled gates add steps when depth exceeds training (OOD-depth mean steps vs mean lower bound)
h1, g2 = {}, {}
for t in TASKS:
    inst_len = pd.concat([entry_instances(cfgs[t], s) for s in ["id_test", "ood_length"]])
    inst_deep = pd.concat([entry_instances(cfgs[t], s) for s in ["ood_depth", "ood_both"]])
    res = g2_check(inst_deep, inst_len)
    od = rows[t][rows[t].split == "ood_depth"].groupby("entry")[["mean_steps", "mean_lb"]].mean()
    res = res.merge(od, left_on="entry", right_index=True, how="left")
    h1[t] = res; display(res.round(3))
    g2[t] = bool(res.shows_premature.any() or res.shows_leak.any())
print("G2 (some baseline family shows premature stopping or length leakage, CI excluding 0):", g2)

,entry,family,premature,premature_lo,premature_hi,premature_err,premature_err_lo,premature_err_hi,acc,leak,leak_lo,leak_hi,shows_premature,premature_costs_acc,shows_leak,mean_steps,mean_lb
0,max_softmax,posthoc,0.000,0.000,0.000,0.000,0.000,0.000,0.999,0.028,0.000,0.048,False,False,False,13.496,13.496
1,entropy,posthoc,0.000,0.000,0.000,0.000,0.000,0.000,0.999,0.000,-0.000,0.000,False,False,False,13.496,13.496
2,stability,posthoc,0.663,0.651,0.675,0.001,0.000,0.002,0.998,0.408,0.387,0.427,True,False,True,5.227,13.496
3,kl,posthoc,0.663,0.651,0.675,0.001,0.000,0.002,0.998,0.393,0.373,0.413,True,False,True,5.320,13.496
4,hidden_change,posthoc,0.333,0.321,0.345,0.191,0.181,0.201,0.809,-0.572,-0.585,-0.558,True,True,False,14.081,13.496
5,max_conf_fan,posthoc,0.000,0.000,0.000,0.000,0.000,0.000,1.000,0.000,0.000,0.000,False,False,False,36.000,13.496
6,trm_post,posthoc,0.834,0.824,0.843,0.170,0.161,0.179,0.830,0.184,0.158,0.211,True,True,True,5.382,13.496
7,act,joint,1.000,1.000,1.000,0.215,0.205,0.226,0.785,0.183,0.157,0.206,True,True,True,2.440,13.496
8,ponder,joint,0.834,0.824,0.843,0.178,0.168,0.187,0.822,0.325,0.301,0.348,True,True,True,6.471,13.496
9,trm_joint,joint,0.831,0.821,0.840,0.227,0.217,0.237,0.773,0.296,0.272,0.319,True,True,True,4.248,13.496


,entry,family,premature,premature_lo,premature_hi,premature_err,premature_err_lo,premature_err_hi,acc,leak,leak_lo,leak_hi,shows_premature,premature_costs_acc,shows_leak,mean_steps,mean_lb
0,max_softmax,posthoc,0.933,0.927,0.939,0.647,0.635,0.659,0.343,0.008,-0.011,0.029,True,True,False,10.286,13.335
1,entropy,posthoc,0.995,0.993,0.996,0.765,0.755,0.776,0.235,0.008,-0.011,0.029,True,True,False,9.411,13.335
2,stability,posthoc,0.860,0.851,0.869,0.580,0.567,0.592,0.355,0.331,0.314,0.347,True,True,True,10.747,13.335
3,kl,posthoc,0.423,0.410,0.435,0.073,0.067,0.080,0.816,0.008,-0.011,0.029,True,True,False,12.984,13.335
4,hidden_change,posthoc,0.874,0.866,0.882,0.573,0.561,0.586,0.421,-0.273,-0.292,-0.251,True,True,False,11.039,13.335
5,max_conf_fan,posthoc,0.000,0.000,0.000,0.000,0.000,0.000,0.950,0.000,0.000,0.000,False,False,False,36.000,13.335
6,trm_post,posthoc,1.000,1.000,1.000,0.786,0.775,0.796,0.214,0.106,0.074,0.141,True,True,True,9.071,13.335
7,act,joint,0.984,0.980,0.987,0.338,0.326,0.350,0.662,0.259,0.242,0.279,True,True,True,11.376,13.335
8,ponder,joint,0.509,0.497,0.522,0.314,0.302,0.325,0.645,0.265,0.246,0.284,True,True,True,13.110,13.335
9,trm_joint,joint,0.951,0.945,0.956,0.388,0.376,0.400,0.606,0.231,0.212,0.251,True,True,True,11.277,13.335


G2 (some baseline family shows premature stopping or length leakage, CI excluding 0): {'addition': True, 'parity': True}


In [8]:
# Fig. 4: steps against depth (OOD-depth + ID) and against length (ID + OOD-length), addition
show = ["oracle_lb", "max_softmax", "stability", "kl", "act", "ponder", "v1gate", "trm_joint", "done_prop_post", "done_joint"]
fig, axes = plt.subplots(len(TASKS), 2, figsize=(7.0, 2.4 * len(TASKS)), squeeze=False)
for i, t in enumerate(TASKS):
    deep = pd.concat([entry_instances(cfgs[t], s, show) for s in ["id_test", "ood_depth", "ood_both"]])
    longi = pd.concat([entry_instances(cfgs[t], s, show) for s in ["id_test", "ood_length"]])
    for ax, inst, col, bins in [(axes[i, 0], deep, "depth", [0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 32, 48, 64, 96, 128]),
                                (axes[i, 1], longi, "length", [4, 8, 12, 16, 20, 64, 96, 128, 192, 256])]:
        g = binned(inst, col, bins)
        for e in show:
            ge = g[g.entry == e]
            if ge.empty: continue
            fam = ENTRIES[e][2]
            ax.plot(ge.x, ge.steps, marker=".", lw=1.2 if fam == "ours" else 0.8, ls="-" if fam != "reference" else ":",
                    color=FAMILY_COLORS[fam], alpha=1.0 if fam in ("ours", "reference") else 0.6, label=ENTRIES[e][3])
        ax.set_xscale("log", base=2); ax.set_yscale("log", base=2)
        ax.set_xlabel("depth $d^*$" if col == "depth" else "length L"); ax.set_ylabel("steps used"); ax.set_title(t)
axes[0, 0].legend(fontsize=5.5, frameon=False, ncol=2)
WRITTEN.append(savefig(fig, "fig4_steps_vs_depth_length"))

In [9]:
# Fig. 5: accuracy-versus-compute curves on OOD-depth (every threshold of every rule)
fig, axes = plt.subplots(1, len(TASKS), figsize=(7.0, 2.6), squeeze=False)
for ax, t in zip(axes[0], TASKS):
    cur = curve_rows(finished(cfgs[t]))
    cur = cur[cur.split == "ood_depth"]
    for e, (method, rule, fam, label) in ENTRIES.items():
        sub = cur[(cur.method == method) & (cur.rule == rule)]
        if sub.empty or e in ("oracle_iter",): continue
        g = sub.groupby("threshold", dropna=False)[["acc", "mean_steps"]].mean().sort_values("mean_steps")
        ax.plot(g.mean_steps, g.acc, marker="." if len(g) > 1 else "*", ms=4 if len(g) > 1 else 8,
                lw=1.2 if fam == "ours" else 0.7, color=FAMILY_COLORS[fam], alpha=0.9 if fam != "posthoc" else 0.6,
                label=label if fam in ("ours", "reference") or e in ("max_softmax", "act", "ponder") else None)
    ax.set_xlabel("mean steps"); ax.set_ylabel("accuracy"); ax.set_title(f"{t}, OOD-depth"); ax.set_xscale("log")
axes[0, 0].legend(fontsize=5.5, frameon=False)
WRITTEN.append(savefig(fig, "fig5_tradeoff"))

In [10]:
# Fig. 6: premature-stop rate against depth
fig, axes = plt.subplots(1, len(TASKS), figsize=(7.0, 2.4), squeeze=False)
for ax, t in zip(axes[0], TASKS):
    inst = pd.concat([entry_instances(cfgs[t], s, show) for s in ["id_test", "ood_depth", "ood_both"]])
    inst["premature"] = (inst.steps < inst.lb).astype(float)
    g = binned(inst, "depth", [0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 32, 48, 64, 96, 128], value="premature")
    for e in show:
        ge = g[g.entry == e]
        if ge.empty or e == "oracle_lb": continue
        fam = ENTRIES[e][2]
        ax.plot(ge.x, ge.premature, marker=".", color=FAMILY_COLORS[fam], lw=1.2 if fam == "ours" else 0.7, label=ENTRIES[e][3])
    ax.axvline(10, color="grey", ls=":", lw=0.8)
    ax.set_xscale("log", base=2); ax.set_xlabel("depth $d^*$"); ax.set_ylabel("premature-stop rate"); ax.set_title(t)
axes[0, 0].legend(fontsize=5.5, frameon=False, ncol=2)
WRITTEN.append(savefig(fig, "fig6_premature_vs_depth"))

In [11]:
WRITTEN.append(write_json(RESULTS_DIR / "gates" / "g2.json", {"G2": g2, **{t: h1[t].to_dict(orient="records") for t in TASKS}}))

In [12]:
print("Files written by this notebook:")
for p in WRITTEN:
    print(" ", Path(p).relative_to(PROJECT_ROOT))

Files written by this notebook:
  results\gates\tuning.json
  paper\figures\fig4_steps_vs_depth_length.pdf
  paper\figures\fig5_tradeoff.pdf
  paper\figures\fig6_premature_vs_depth.pdf
  results\gates\g2.json
